# Examen de Recuperación de Información: NFCorpus

Se usa `Path.cwd()` para buscar los datos junto al notebook. El bucle de la
siguiente celda toma cada nombre de `ARCHIVOS`, forma su ruta y pregunta con
`is_file()` si existe. Si algún valor sale `False`, falta ese archivo en la
carpeta desde la que se ejecutó Jupyter.

In [2]:
from pathlib import Path

CARPETA = Path.cwd()

ARCHIVOS = [
    "corpus-00000-of-00001.parquet",
    "queries-00000-of-00001.parquet",
    "train.tsv",
    "dev.tsv",
    "test.tsv",
]

for nombre in ARCHIVOS:
    ruta = CARPETA / nombre
    print(nombre, "presente:", ruta.is_file())

corpus-00000-of-00001.parquet presente: True
queries-00000-of-00001.parquet presente: True
train.tsv presente: True
dev.tsv presente: True
test.tsv presente: True


## Fase 1. Preprocesamiento

El orden de sus operaciones es:

1. `casefold()` unifica mayúsculas y minúsculas, NFKD separa letras y
   marcas de acento. El bucle `for caracter` conserva las letras base y
   descarta marcas combinantes.
2. `re.findall(r"[a-z0-9]+", ...)` produce una lista de palabras y números.
   Por ejemplo, la puntuación deja de ser parte del término, donde una expresión
   con guion puede quedar separada.
3. El bucle `for palabra` omite stopwords frecuentes. Se preservan `no`,
   `not`, `nor`, `never` y `without` porque una negación puede invertir
   el sentido de una afirmación biomédica.
4. `STEMMER.stem(palabra)` reduce variantes inglesas a una raíz. Por
   ejemplo, `studies`, `study` y `studying` terminan como `studi`.

Se utilizó Snowball porque NFCorpus está en inglés y las consultas pueden usar
una forma gramatical distinta de la del artículo. Dejar cada forma separada
perdería coincidencias, entonces Snowball proporciona reglas ligeras y deterministas
que funcionan sin descargar un modelo gramatical, una lematización en cambio con
análisis de contexto añadiría recursos y más pasos.

In [3]:
"""Motor de recuperación NFCorpus"""

import csv
import math
import re
import unicodedata
from collections import Counter
import numpy as np
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer


# Las negaciones cambian el sentido de consultas biomédicas: no deben eliminarse.
NEGACIONES = {"no", "not", "nor", "never", "without"}
STOPWORDS = set(ENGLISH_STOP_WORDS) - NEGACIONES
STEMMER = SnowballStemmer("english")


def tokenizar(texto):
    """Normaliza, separa palabras, quita stopwords y aplica stemming inglés."""
    tokens = []
    # NFKD conserva letras acentuadas como su letra base antes del filtrado.
    texto_normalizado = unicodedata.normalize("NFKD", texto.casefold())
    caracteres = []

    for caracter in texto_normalizado:
        if not unicodedata.combining(caracter):
            caracteres.append(caracter)

    texto_normalizado = "".join(caracteres)
    palabras = re.findall(r"[a-z0-9]+", texto_normalizado)

    for palabra in palabras:
        if palabra not in STOPWORDS:
            raiz = STEMMER.stem(palabra)
            tokens.append(raiz)

    return tokens

In [4]:
# Ejemplo de tokenizacion
ejemplos = ["Studies of nutritional diseases", "No evidence without treatment"]

for frase in ejemplos:
    print("Texto:", frase)
    print("Tokens:", tokenizar(frase))


Texto: Studies of nutritional diseases
Tokens: ['studi', 'nutrit', 'diseas']
Texto: No evidence without treatment
Tokens: ['no', 'evid', 'without', 'treatment']


## Fase 2. Leer corpus, consultas y qrels

`leer_corpus()` abre el Parquet de documentos y se encarga de conservar el
ID tal y como aparece en el dataset.

Con `leer_consultas()` creamos un diccionario **ID → texto**. Esto permite obtener
rápidamente el texto asociado a cada ID de `test`. `leer_qrels()` recorre
las filas del TSV y construye **ID de consulta → {ID de documento: grado}**.
El grado 1 o 2 se conserva para NDCG

In [5]:
def leer_corpus(ruta):
    """Lee todos los documentos; conserva el identificador original."""
    import pyarrow.parquet as parquet

    tabla = parquet.read_table(ruta, columns=["_id", "title", "text"])
    filas = tabla.to_pylist()
    documentos = []

    for fila in filas:
        documento = {
            "id": str(fila["_id"]),
            "title": fila["title"] or "",
            "text": fila["text"] or "",
        }
        documentos.append(documento)

    return documentos


def leer_consultas(ruta):
    """Lee las consultas; NFCorpus guarda el texto útil en la columna text."""
    import pyarrow.parquet as parquet

    tabla = parquet.read_table(ruta, columns=["_id", "text"])
    consultas = {}

    for fila in tabla.to_pylist():
        consultas[str(fila["_id"])] = fila["text"] or ""

    return consultas


def leer_qrels(ruta):
    """Devuelve relevancias agrupadas por ID de consulta y documento."""
    qrels = {}

    with open(ruta, "r", encoding="utf-8", newline="") as archivo:
        lector = csv.DictReader(archivo, delimiter="\t")

        for fila in lector:
            id_consulta = fila["query-id"]
            id_documento = fila["corpus-id"]
            relevancia = int(fila["score"])

            if id_consulta not in qrels:
                qrels[id_consulta] = {}

            qrels[id_consulta][id_documento] = relevancia

    return qrels

## Fase 3. Vocabulario, índice invertido y matriz TF-IDF

`crear_indice()` recorre los 3.633 documentos completos. En cada iteración:

- Junta una vez título y cuerpo. **No** divide el artículo en fragmentos:
  la unidad que se recupera coincide con la unidad de los qrels.
- `Counter(tokenizar(texto))` cuenta cuántas veces aparece cada término;
  `sum(frecuencias.values())` calcula la longitud en tokens para BM25.
- El bucle `for termino, frecuencia` guarda la lista de documentos que
  contienen ese término: `indice_invertido[termino][numero_documento]`.
  Si un término aparece tres veces en la fila 8, allí se guarda `3`.

`TfidfVectorizer` calcula una matriz con una fila por documento y una columna
por término. Su opción `norm="l2"` deja cada vector con norma 1; así el
producto punto con la consulta normalizada equivale a similitud coseno.
La longitud media se conserva para comparar cada documento con el promedio
en BM25. Se guardan `k1=1.2` (saturación de frecuencia) y `b=0.75`
(corrección por longitud) como parámetros iniciales explícitos.

In [6]:
def crear_indice(documentos):
    """Construye vocabulario, índice invertido y matriz TF-IDF normalizada."""
    textos = []
    indice_invertido = {}
    longitudes = []

    for numero_documento, documento in enumerate(documentos):
        # El título y el cuerpo forman UNA unidad: los qrels usan el ID original.
        texto = documento["title"] + " " + documento["text"]
        textos.append(texto)
        frecuencias = Counter(tokenizar(texto))
        longitudes.append(sum(frecuencias.values()))

        for termino, frecuencia in frecuencias.items():
            if termino not in indice_invertido:
                indice_invertido[termino] = {}

            indice_invertido[termino][numero_documento] = frecuencia

    # L2 hace que el producto punto de dos vectores sea similitud coseno.
    vectorizador = TfidfVectorizer(analyzer=tokenizar, norm="l2", dtype=np.float32)
    matriz_tfidf = vectorizador.fit_transform(textos)

    sistema = {
        "documentos": documentos,
        "indice_invertido": indice_invertido,
        "longitudes": longitudes,
        "longitud_media": sum(longitudes) / len(longitudes),
        "vectorizador": vectorizador,
        "matriz_tfidf": matriz_tfidf,
        "k1": 1.2,
        "b": 0.75,
    }
    return sistema

## Fase 4. Dos rankings independientes

**BM25:** `Counter(tokens_consulta)` entrega cada término de la consulta y
su frecuencia. Para cada uno, `apariciones` contiene **solo** los documentos
donde existe. `idf` disminuye el peso de términos que aparecen en muchos
documentos. El bucle interior suma `aporte` al puntaje del documento usando
frecuencia, longitud, `k1` y `b`. `sorted()` ordena primero por puntaje
descendente y usa el ID original para resolver empates de manera estable.

**TF-IDF:** `transform([consulta])` aplica el mismo vocabulario aprendido
del corpus. El producto de la matriz de documentos y el vector de consulta
produce el coseno. Se conservan puntuaciones positivas, se ordenan y se
toman las primeras `top_k`. Los puntajes de TF-IDF y BM25 tienen escalas
diferentes; se compara el **orden** y las métricas, no sus números brutos.

`buscar()` convierte el ranking interno en la respuesta esperada: posición,
ID original, puntuación, título y los primeros 240 caracteres del cuerpo.
Ese `snippet` es una vista del texto; no crea otra unidad indexada. Cada
número de documento aparece una vez en el ranking, por lo que no hay
duplicados. `top_k` acepta el valor solicitado por el cliente; el 10 de
las métricas se explica en la fase de evaluación.

In [7]:
def buscar_bm25(sistema, consulta, top_k):
    """Acumula BM25 solo sobre listas de documentos con términos de consulta."""
    tokens_consulta = tokenizar(consulta)
    puntuaciones = {}
    total_documentos = len(sistema["documentos"])

    # Counter cuenta repetición de términos de consulta sin una sintaxis compacta.
    for termino, frecuencia_consulta in Counter(tokens_consulta).items():
        apariciones = sistema["indice_invertido"].get(termino, {})
        frecuencia_documental = len(apariciones)

        if frecuencia_documental == 0:
            continue

        idf = math.log(1 + (total_documentos - frecuencia_documental + 0.5)
                       / (frecuencia_documental + 0.5))

        for numero_documento, frecuencia_documento in apariciones.items():
            longitud = sistema["longitudes"][numero_documento]
            ajuste = sistema["k1"] * (
                1 - sistema["b"] + sistema["b"] * longitud / sistema["longitud_media"]
            )
            aporte = idf * frecuencia_documento * (sistema["k1"] + 1)
            aporte = aporte / (frecuencia_documento + ajuste)
            aporte = aporte * frecuencia_consulta

            if numero_documento not in puntuaciones:
                puntuaciones[numero_documento] = 0.0

            puntuaciones[numero_documento] += aporte

    ranking = sorted(
        puntuaciones.items(),
        key=lambda par: (-par[1], sistema["documentos"][par[0]]["id"]),
    )
    return ranking[:top_k]


def buscar_tfidf(sistema, consulta, top_k):
    """Transforma una consulta libre y compara con todos los documentos."""
    vector_consulta = sistema["vectorizador"].transform([consulta])
    puntuaciones = (sistema["matriz_tfidf"] @ vector_consulta.T).toarray().ravel()
    ranking = []

    for numero_documento, puntuacion in enumerate(puntuaciones):
        if puntuacion > 0:
            ranking.append((numero_documento, float(puntuacion)))

    ranking.sort(
        key=lambda par: (-par[1], sistema["documentos"][par[0]]["id"])
    )
    return ranking[:top_k]


def buscar(sistema, consulta, modelo, top_k):
    """Entrega IDs originales, puntuaciones y texto para el servicio JSON."""
    if modelo == "bm25":
        ranking = buscar_bm25(sistema, consulta, top_k)
    else:
        ranking = buscar_tfidf(sistema, consulta, top_k)

    resultados = []

    for posicion, par in enumerate(ranking, start=1):
        numero_documento, puntuacion = par
        documento = sistema["documentos"][numero_documento]
        resultado = {
            "rank": posicion,
            "document_id": documento["id"],
            "score": float(puntuacion),
            "title": documento["title"],
            "snippet": documento["text"][:240],
        }
        resultados.append(resultado)

    return resultados

## Fase 5. Métricas por consulta y promedio del sistema

`metricas_consulta()` toma los diez primeros documentos y los qrels de
**una** consulta. El primer bucle cuenta todos los documentos conocidos
como relevantes. El segundo examina cada resultado: suma un acierto si
su grado es positivo y aporta `(2**grado - 1) / log2(posición + 1)` a DCG.


`evaluar()` recorre cada ID de `qrels_test`, recupera su texto, busca diez
resultados y guarda una fila de métricas. Luego otro bucle suma cada
columna y divide por la cantidad de consultas. El `10` se eligió porque
el enunciado exige estas métricas `@10`; en el servicio `top_k` puede ser
distinto. Una consulta libre sin qrels puede recuperarse, pero no tiene
relevancias conocidas con las que calcular estas métricas.

In [8]:
def metricas_consulta(resultados, relevancias, k=10):
    """Calcula P/R binarios y NDCG con grados de relevancia originales."""
    recuperados = resultados[:k]
    positivos = 0
    total_relevantes = 0
    dcg = 0.0

    for grado in relevancias.values():
        if grado > 0:
            total_relevantes += 1

    for posicion, resultado in enumerate(recuperados, start=1):
        grado = relevancias.get(resultado["document_id"], 0)

        if grado > 0:
            positivos += 1

        dcg += (2 ** grado - 1) / math.log2(posicion + 1)

    grados_ideales = sorted(relevancias.values(), reverse=True)[:k]
    idcg = 0.0

    for posicion, grado in enumerate(grados_ideales, start=1):
        idcg += (2 ** grado - 1) / math.log2(posicion + 1)

    precision = positivos / k
    recall = positivos / total_relevantes if total_relevantes > 0 else 0.0
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return {"precision@10": precision, "recall@10": recall, "ndcg@10": ndcg}


def evaluar(sistema, consultas, qrels, modelo):
    """Evalúa cada consulta de test y promedia las métricas sin saltarse ninguna."""
    filas = []

    for id_consulta, relevancias in qrels.items():
        texto = consultas[id_consulta]
        resultados = buscar(sistema, texto, modelo, 10)
        metricas = metricas_consulta(resultados, relevancias)
        fila = {"query_id": id_consulta}
        fila.update(metricas)
        filas.append(fila)

    promedio = {}

    for nombre in ("precision@10", "recall@10", "ndcg@10"):
        suma = 0.0

        for fila in filas:
            suma += fila[nombre]

        promedio[nombre] = suma / len(filas)

    return filas, promedio

## Fase 6. Prueba de búsqueda y evaluación sobre `test`

Primero se consulta una frase libre con los dos modelos para comprobar que
generan órdenes independientes. El bucle exterior cambia el modelo; el
interior muestra puesto, ID, puntuación y título de los tres primeros.
La puntuación es propia de cada modelo y no debe compararse directamente
entre TF-IDF y BM25.

### Ejecutar las funciones del notebook

Las celdas anteriores ya definieron en este notebook la tokenización, la lectura, la indexación, los dos rankings y las métricas. Ahora se cargan los datos y se construye `sistema`. El código puede ejecutarse sin importar `motor_ri.py`; solo necesita los archivos del corpus y qrels en esta carpeta.

In [9]:
# Cargamos los documentos y consultas completos y las tres tablas de qrels.
# Solo test se usa para la evaluación final; train y dev se muestran para
# conocer las particiones disponibles.
documentos = leer_corpus(CARPETA / "corpus-00000-of-00001.parquet")
consultas = leer_consultas(CARPETA / "queries-00000-of-00001.parquet")
qrels_train = leer_qrels(CARPETA / "train.tsv")
qrels_dev = leer_qrels(CARPETA / "dev.tsv")
qrels_test = leer_qrels(CARPETA / "test.tsv")

print("Documentos:", len(documentos))
print("Consultas disponibles:", len(consultas))
print("Consultas con qrels train:", len(qrels_train))
print("Consultas con qrels dev:", len(qrels_dev))
print("Consultas con qrels test:", len(qrels_test))
print("Primer ID original:", documentos[0]["id"])

Documentos: 3633
Consultas disponibles: 3237
Consultas con qrels train: 2590
Consultas con qrels dev: 324
Consultas con qrels test: 323
Primer ID original: MED-10


In [10]:
# Esta llamada recorre cada documento una vez para construir las estructuras.
# Los tres print permiten comprobar cantidad de términos, forma de la
# matriz y longitud media antes de buscar.
sistema = crear_indice(documentos)
print("Términos del vocabulario:", len(sistema["indice_invertido"]))
print("Matriz TF-IDF:", sistema["matriz_tfidf"].shape)
print("Longitud media (tokens):", round(sistema["longitud_media"], 2))

Términos del vocabulario: 18862
Matriz TF-IDF: (3633, 18862)
Longitud media (tokens): 161.11


In [11]:
# La consulta es libre: no necesita aparecer en los qrels.
# El bucle exterior ejecuta cada modelo; el interior muestra el ranking.
consulta_libre = "nutrition and heart disease prevention"

for modelo in ("tfidf", "bm25"):
    resultados = buscar(sistema, consulta_libre, modelo, 5)
    print("\nModelo:", modelo, "| Consulta:", consulta_libre)

    for resultado in resultados:
        print("Puesto:", resultado["rank"])
        print("ID original:", resultado["document_id"])
        print("Puntuación:", round(resultado["score"], 4))
        print("Título:", resultado["title"])
        print("Fragmento:", resultado["snippet"][:140])
        print()



Modelo: tfidf | Consulta: nutrition and heart disease prevention
Puesto: 1
ID original: MED-4315
Puntuación: 0.3157
Título: Differences in metal content of the heart muscle in death from ischemic heart disease.
Fragmento: In a group of patients dying suddenly from ischemic heart disease, the uninfarcted heart muscle contained significantly lower concentrations

Puesto: 2
ID original: MED-4496
Puntuación: 0.2669
Título: The effect of fruit and vegetable intake on risk for coronary heart disease.
Fragmento: BACKGROUND: Many constituents of fruits and vegetables may reduce the risk for coronary heart disease, but data on the relationship between 

Puesto: 3
ID original: MED-4255
Puntuación: 0.2596
Título: Resolving the Coronary Artery Disease Epidemic Through Plant-Based Nutrition.
Fragmento: The world's advanced countries have easy access to plentiful high-fat food; ironically, it is this rich diet that produces atherosclerosis. 

Puesto: 4
ID original: MED-4613
Puntuación: 0.2596
Títul

La evaluación usa **todas las consultas que aparecen en `test.tsv`**: 323
IDs, mostrados en la siguiente salida. `metricas_por_consulta[modelo]`
conserva las 323 filas calculadas para revisar una consulta concreta;
`promedios[modelo]` conserva los tres promedios. Se imprimen las primeras
tres filas como ejemplo y el promedio completo de cada modelo.

In [12]:
# Todas las claves de qrels_test son consultas evaluables.
# Guardamos las métricas de cada consulta y el promedio de cada modelo.
ids_test = list(qrels_test.keys())
print("Cantidad de consultas test utilizadas:", len(ids_test))
print("IDs test utilizados:")
print(", ".join(ids_test))

metricas_por_consulta = {}
promedios = {}

for modelo in ("tfidf", "bm25"):
    filas, promedio = evaluar(sistema, consultas, qrels_test, modelo)
    metricas_por_consulta[modelo] = filas
    promedios[modelo] = promedio
    print("\n", modelo, "promedio en test:", promedio)
    print("Primeras 3 consultas y sus métricas:")

    for fila in filas[:3]:
        print(fila)

Cantidad de consultas test utilizadas: 323
IDs test utilizados:
PLAIN-2, PLAIN-12, PLAIN-23, PLAIN-33, PLAIN-44, PLAIN-56, PLAIN-68, PLAIN-78, PLAIN-91, PLAIN-102, PLAIN-112, PLAIN-123, PLAIN-133, PLAIN-143, PLAIN-153, PLAIN-165, PLAIN-175, PLAIN-186, PLAIN-196, PLAIN-207, PLAIN-217, PLAIN-227, PLAIN-238, PLAIN-248, PLAIN-259, PLAIN-270, PLAIN-280, PLAIN-291, PLAIN-307, PLAIN-320, PLAIN-332, PLAIN-344, PLAIN-358, PLAIN-371, PLAIN-383, PLAIN-395, PLAIN-407, PLAIN-418, PLAIN-430, PLAIN-441, PLAIN-457, PLAIN-468, PLAIN-478, PLAIN-488, PLAIN-499, PLAIN-510, PLAIN-520, PLAIN-531, PLAIN-541, PLAIN-551, PLAIN-561, PLAIN-571, PLAIN-583, PLAIN-593, PLAIN-603, PLAIN-613, PLAIN-623, PLAIN-634, PLAIN-645, PLAIN-660, PLAIN-671, PLAIN-681, PLAIN-691, PLAIN-701, PLAIN-711, PLAIN-721, PLAIN-731, PLAIN-741, PLAIN-751, PLAIN-761, PLAIN-771, PLAIN-782, PLAIN-792, PLAIN-806, PLAIN-817, PLAIN-827, PLAIN-838, PLAIN-850, PLAIN-872, PLAIN-882, PLAIN-892, PLAIN-902, PLAIN-913, PLAIN-924, PLAIN-934, PLAIN-946, 

La tabla textual siguiente calcula `BM25 - TF-IDF` para cada métrica. Una
diferencia positiva favorece a BM25 en esa medida. Como ambos modelos
fueron evaluados con los mismos documentos, consultas y qrels de `test`,
la comparación usa una base común.

In [13]:
# Recorremos los tres nombres de métricas para comparar la misma
# medida entre TF-IDF y BM25, sobre las mismas consultas.
print("Comparación final sobre las mismas consultas de test:")

for nombre in ("precision@10", "recall@10", "ndcg@10"):
    valor_tfidf = promedios["tfidf"][nombre]
    valor_bm25 = promedios["bm25"][nombre]
    diferencia = valor_bm25 - valor_tfidf
    print(nombre, "TF-IDF:", round(valor_tfidf, 4),
          "BM25:", round(valor_bm25, 4),
          "diferencia BM25 - TF-IDF:", round(diferencia, 4))

Comparación final sobre las mismas consultas de test:
precision@10 TF-IDF: 0.2282 BM25: 0.2378 diferencia BM25 - TF-IDF: 0.0096
recall@10 TF-IDF: 0.1512 BM25: 0.1562 diferencia BM25 - TF-IDF: 0.005
ndcg@10 TF-IDF: 0.3115 BM25: 0.3297 diferencia BM25 - TF-IDF: 0.0182


### Interpretación de los resultados de test

| Métrica | TF-IDF | BM25 | Ventaja de BM25 |
|---|---:|---:|---:|
| Precision@10 | 0,2282 | **0,2378** | +0,0096 |
| Recall@10 | 0,1512 | **0,1562** | +0,0050 |
| NDCG@10 | 0,3115 | **0,3297** | +0,0182 |

Se evaluaron las mismas 323 consultas con ambos modelos y los qrels de `test`. BM25 obtuvo Precision@10 = **0,2378**, Recall@10 = **0,1562** y NDCG@10 = **0,3297**; TF-IDF obtuvo **0,2282**, **0,1512** y **0,3115**, respectivamente. BM25 supera a TF-IDF en las tres métricas, aunque las diferencias son pequeñas: aproximadamente 0,0096 en precisión, 0,0050 en recall y 0,0182 en NDCG.

Una Precision@10 de 0,2378 significa que BM25 recupera, en promedio, 2,38 documentos relevantes entre los diez primeros. Un Recall@10 de 0,1562 significa que esos diez resultados contienen alrededor del 15,62 % de todos los documentos relevantes conocidos para cada consulta, al promediar las consultas. NDCG@10 considera además la posición y el grado de relevancia: su valor mayor para BM25 indica un orden ligeramente mejor en los primeros diez puestos. Estas cifras describen el comportamiento medido en `test` con `k=10`, no convierten una consulta libre sin qrels en evaluable.

TF-IDF encontró 737 aciertos entre los primeros diez puestos de todas las consultas; BM25 encontró 768, es decir, 31 aciertos consulta-documento adicionales.

La comparación por consulta confirma ese matiz. En Precision@10, BM25 gana en 53 consultas, TF-IDF en 31 y empatan en 239. En NDCG@10, BM25 gana en 106, TF-IDF en 55 y empatan en 162. Esto sugiere que la ventaja de BM25 se aprecia más en qué documentos coloca primero y con qué grado de relevancia que solo en la cantidad de aciertos.

Un ejemplo es PLAIN-2 (Do Cholesterol Statin Drugs Cause Breast Cancer?). TF-IDF encuentra 8 relevantes en el top 10 y BM25 7; por ello TF-IDF tiene mejor Precision@10 en esa consulta. Sin embargo, BM25 obtiene mayor NDCG@10 (0,7806 frente a 0,7652): según los grados de los qrels, ordenó mejor los resultados más valiosos en los puestos altos. Por eso ninguna métrica, aislada, cuenta toda la historia.

El Recall@10 promedio de BM25, 0,1562, debe leerse teniendo en cuenta el corte en diez resultados. Los qrels de test contienen 38,19 documentos relevantes por consulta en promedio, con mediana de 16, de modo que para muchas consultas es imposible mostrar todos los relevantes en solo diez puestos. Además, el resultado es un promedio de recall por consulta, no el porcentaje de todos los qrels reunidos en una sola lista.

En conclusión BM25 obtuvo resultados ligeramente mejors porque ajusta la puntuación según la longitud del documento y limita el beneficio de repetir muchas veces un término, algo útil en este corpus, cuyos textos procesados varían entre 11 y 1.035 tokens. La diferencia es moderada: ambos modelos empataron en Precision@10 en 239 consultas.